In [8]:
# --- 1. SETUP ---
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

StatementMeta(, 9b8ce755-0430-4b41-9449-5ca8fd278900, 10, Finished, Available, Finished, False)

In [2]:

# --- 2. CONFIGURATION ---
SUBSTATIONS = ["Auckland_North", "Hamilton_Central", "Wellington_South", "Christchurch_West"]
ASSET_TYPES = ["Power_Transformer", "HV_Circuit_Breaker", "Voltage_Regulator"]
ASSET_COUNT = 30
current_time = datetime.now()

print("🛠️ Generating synthetic Omexom asset data...")

StatementMeta(, 9b8ce755-0430-4b41-9449-5ca8fd278900, 4, Finished, Available, Finished, False)

🛠️ Generating synthetic Omexom asset data...


In [13]:
# --- 3. LOGIC ---
data = []
for i in range(ASSET_COUNT):
    asset_id = f"OMX-{1000 + i}"
    asset_type = str(np.random.choice(ASSET_TYPES))
    substation = str(np.random.choice(SUBSTATIONS))
    install_date = (current_time - timedelta(days=np.random.randint(365, 7300))).strftime('%Y-%m-%d')
    
    for d in range(10):
        timestamp = (current_time - timedelta(days=d)).strftime('%Y-%m-%d %H:%M:%S')
        temp = float(np.random.normal(50, 5))     
        load_pct = float(np.random.normal(60, 10)) 
        oil_level = float(np.random.uniform(90, 100))
        
        # Inject Anomaly: OMX-1002 is overheating
        # Added float() casting to ensure compatibility with Spark DoubleType
        if asset_id == "OMX-1002" and d < 3:
            temp += float(45)
        
        # Anomaly: OMX-1015 has low oil level
        if asset_id == "OMX-1015":
            oil_level = float(45 - d) 

        data.append([asset_id, asset_type, substation, install_date, timestamp, temp, load_pct, oil_level])


StatementMeta(, 9b8ce755-0430-4b41-9449-5ca8fd278900, 15, Finished, Available, Finished, False)

In [11]:
display(data)

StatementMeta(, 9b8ce755-0430-4b41-9449-5ca8fd278900, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f1c95132-5f57-4f7b-b6f2-14f3bfe90642)

In [14]:
# --- 4. SAVE TO LAKEHOUSE ---
# Explicitly defining the schema to prevent Type Inference errors
schema = StructType([
    StructField("AssetID", StringType(), True),
    StructField("AssetType", StringType(), True),
    StructField("Substation", StringType(), True),
    StructField("InstallationDate", StringType(), True),
    StructField("ReadingTimestamp", StringType(), True),
    StructField("Temperature_C", DoubleType(), True),
    StructField("Load_Pct", DoubleType(), True),
    StructField("OilLevel_Pct", DoubleType(), True)
])

# Create Spark DataFrame with the explicit schema
spark_df = spark.createDataFrame(data, schema=schema)

# Write to the 'Files' directory in your Lakehouse
# Note: Ensure you have a Lakehouse attached to this notebook
path = "Files/Landing/omexom_raw_assets.csv"
spark_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(path)

print(f"✅ Step 1 Success: Data generated and saved to {path}")
print("🚨 Demo Note: Asset OMX-1002 is failing (Heat) and OMX-1015 is failing (Oil).")

StatementMeta(, 9b8ce755-0430-4b41-9449-5ca8fd278900, 16, Finished, Available, Finished, False)

✅ Step 1 Success: Data generated and saved to Files/Landing/omexom_raw_assets.csv
🚨 Demo Note: Asset OMX-1002 is failing (Heat) and OMX-1015 is failing (Oil).
